# Credit VaR Model Comparison — CreditMetrics, KMV, and BIS/Basel IRB

This notebook runs all three credit risk models in this repository side by side:

| Model | Method | File |
|---|---|---|
| **CreditMetrics** | Monte Carlo, rating migration | `creditmetrics-var/creditmetrics_model.py` |
| **KMV / Merton** | Monte Carlo, structural (asset-value) default | `kmv-model/kmv_model.py` |
| **BIS / Basel IRB** | Closed-form regulatory formula | `bis-irb-model/bis_irb_model.py` |

## ⚠️ Important: these three models are not fully comparable at an arbitrary shared alpha

- **CreditMetrics** and **KMV** are both Monte Carlo simulations that produce an *empirical loss
  distribution* across the whole portfolio. Their confidence level (`alpha`) is a percentile you
  choose freely, so **these two can be directly compared at the same alpha** (this notebook does
  that below).
- **BIS/Basel IRB** is a **closed-form regulatory formula** applied *per exposure*, not a portfolio
  Monte Carlo simulation. Its confidence level is **fixed by Basel regulation at 99.9%**
  (`norm.ppf(0.999)` is hardcoded into the formula) — it is not a free parameter you can vary to
  "match" the other two models' alpha. Running BIS at a different confidence level would not be a
  Basel-compliant capital number; it would just be a different formula.
- Basel's IRB formula is also **calibrated at the single-exposure level** (PD, LGD in), producing a
  capital *rate* per exposure — it does not natively produce a joint multi-firm portfolio loss
  distribution the way the two Monte Carlo models do, so a firm-by-firm "total portfolio BIS VaR"
  requires summing exposure-level K \* EAD, which is an additional aggregation step, not part of
  the base formula.

**Practical takeaway:** set `ALPHA` below once, and it drives both CreditMetrics and KMV consistently.
BIS is reported alongside at its fixed 99.9% for reference, with a clear label — not silently
recomputed at your chosen `ALPHA`.


In [1]:
# --- Setup: make the sibling model folders importable ---
import sys, os

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
for subfolder in ["creditmetrics-var", "kmv-model", "bis-irb-model"]:
    path = os.path.join(REPO_ROOT, subfolder)
    if path not in sys.path:
        sys.path.append(path)

import numpy as np
import pandas as pd

print("Python path configured. Repo root:", REPO_ROOT)


Python path configured. Repo root: /workspaces/credit-risk-models


## Shared configuration

Change `ALPHA` here to re-run **CreditMetrics** and **KMV** at any confidence level you like
(e.g. `0.95`, `0.99`, `0.999`). This is the one knob referenced in the repo README for
"comparing CVaR at the same alpha."


In [2]:
# --- Shared parameters ---
ALPHA = 0.999          # applies to CreditMetrics + KMV (Monte Carlo models only)
N_SIMULATIONS = 1_000_000   # reduce for a faster local run, e.g. 100_000

BIS_FIXED_ALPHA = 0.999  # NOT adjustable — hardcoded into Basel's IRB formula


## 1. CreditMetrics — Rating Migration Model

In [3]:
from creditmetrics_model import run_full_model as run_creditmetrics

cm_result = run_full_model = run_creditmetrics(alpha=ALPHA, n_simulations=N_SIMULATIONS)

print(f"--- CreditMetrics Credit VaR ---")
print(f"Confidence Level (Alpha): {cm_result['alpha']:.1%}")
print(f"Credit VaR:               {cm_result['Credit VaR']:,.2f}")
print(f"Expected Loss (EL):       {cm_result['Expected Loss']:,.2f}")
print(f"Economic Capital (EC):    {cm_result['Economic Capital']:,.2f}")


--- CreditMetrics Credit VaR ---
Confidence Level (Alpha): 99.9%
Credit VaR:               2,563,500.42
Expected Loss (EL):       188,073.14
Economic Capital (EC):    2,375,427.28


In [4]:
# Full scenario-level detail table (probability, loss, cumulative probability)
cm_result['detail_table'].sort_values('Loss', ascending=False).head(15)


,index,Probability,Loss,Cum_Prob
582,"(S,D,D,D,D)",0.000004,3.799492e+06,1.000000
581,"(S,B,D,D,D)",0.000002,3.584131e+06,0.999996
580,"(S,BB,D,D,D)",0.000014,3.583642e+06,0.999994
579,"(S,D,CCC,D,D)",0.000001,3.255821e+06,0.999980
578,"(S,BB,CCC,D,D)",0.000007,3.039971e+06,0.999979
577,"(S,B,B,D,D)",0.000001,3.024068e+06,0.999972
576,"(S,BB,B,D,D)",0.000002,3.023579e+06,0.999971
575,"(D,BB,D,A,D)",0.000002,2.883500e+06,0.999969
574,"(D,B,D,AA,D)",0.000001,2.875404e+06,0.999967
573,"(D,BB,D,AA,D)",0.000005,2.874916e+06,0.999966


## 2. KMV / Merton — Structural Default Model

In [5]:
from kmv_model import PORTFOLIO as KMV_PORTFOLIO, run_kmv_simulation, get_scenario_summary

results = run_kmv_simulation(
    KMV_PORTFOLIO, N_SIMULATIONS, confidence_level=ALPHA
)

print(f"Expected Loss: {results['expected_loss']:,.2f}")
print(f"Portfolio Credit VaR ({ALPHA*100}%): {results['p_var']:,.2f}")
print(f"Economic Capital: {results['economic_capital']:,.2f}")
print(f"Portfolio CVaR (Expected Shortfall): {results['p_cvar']:,.2f}")


Expected Loss: 30,570.93
Portfolio Credit VaR (99.9%): 79,200.00
Economic Capital: 48,629.07
Portfolio CVaR (Expected Shortfall): 79,200.00


In [6]:
kmv_scenario_table = get_scenario_summary(KMV_PORTFOLIO, results['default_matrix'], results['total_loss'])
kmv_scenario_table


,Scenario,Avg_Loss,Probability,Cumulative_Prob
7,"(S,S,S)",0.0,0.281482,0.281482
6,"(S,S,D)",18000.0,0.033039,0.314521
5,"(S,D,S)",25200.0,0.016342,0.330863
3,"(D,S,S)",36000.0,0.452660,0.783523
4,"(S,D,D)",43200.0,0.004583,0.788106
2,"(D,S,D)",54000.0,0.074962,0.863068
1,"(D,D,S)",61200.0,0.101239,0.964307
0,"(D,D,D)",79200.0,0.035693,1.000000


## 3. BIS / Basel IRB — Closed-Form Regulatory Capital

Reported **per exposure** at the fixed 99.9% Basel confidence level. This is NOT re-run at `ALPHA`
above — see the explanation at the top of this notebook for why that would not be meaningful.

Below applies the formula to each KMV-portfolio firm as an illustration (using each firm's
LGD and an approximate 1-year default probability implied by its asset/debt/volatility inputs
would require a separate PD estimation step — here we instead demonstrate the formula directly
with example PD/LGD inputs, consistent with the original script).


In [7]:
from bis_irb_model import get_single_factor_bis, BASEL_CONFIDENCE_LEVEL

# Example single-exposure illustration (LGD=40%, PD=10%) — replace with your own PD/LGD
# per exposure to size regulatory capital for a specific facility.
bis_result = get_single_factor_bis(lgd=0.40, pd=0.10)

print(f"--- BIS/Basel IRB Capital (fixed {BASEL_CONFIDENCE_LEVEL:.1%} confidence) ---")
print(f"Capital Requirement (K): {bis_result['Capital Requirement']:.6f}  (as a fraction of EAD)")
print(f"Correlation:             {bis_result['Correlation']:.6f}")
print(f"Expected Loss:           {bis_result['Expected Loss']:.6f}")
print(f"CVaR (K + EL):           {bis_result['CVaR']:.6f}")


--- BIS/Basel IRB Capital (fixed 99.9% confidence) ---
Capital Requirement (K): 0.124978  (as a fraction of EAD)
Correlation:             0.120809
Expected Loss:           0.040000
CVaR (K + EL):           0.164978


## 4. Side-by-Side Summary

CreditMetrics and KMV are shown at the **same chosen `ALPHA`**. BIS is shown separately at its
**fixed 99.9%** — they are labeled differently on purpose, not merged into one misleading row.


In [8]:
summary = pd.DataFrame([
    {
        "Model": "CreditMetrics (Monte Carlo)",
        "Alpha": f"{cm_result['alpha']:.1%}",
        "Credit VaR": cm_result['Credit VaR'],
        "Expected Loss": cm_result['Expected Loss'],
        "Economic Capital": cm_result['Economic Capital'],
    },
    {
        "Model": "KMV / Merton (Monte Carlo)",
        "Alpha": f"{ALPHA:.1%}",
        "Credit VaR": results['p_var'],
        "Expected Loss": results['expected_loss'],  # EL not directly comparable here; see notes
        "Economic Capital": results['economic_capital'],
    },
    {
        "Model": "BIS/Basel IRB (closed-form, per exposure)",
        "Alpha": f"{BASEL_CONFIDENCE_LEVEL:.1%}",
        "Credit VaR": bis_result['CVaR'],
        "Expected Loss": bis_result['Expected Loss'],
        "Economic Capital": bis_result['Capital Requirement'],
    },
])

summary


,Model,Alpha,Credit VaR,Expected Loss,Economic Capital
0,CreditMetrics (Monte Carlo),99.9%,2.563500e+06,188073.136692,2.375427e+06
1,KMV / Merton (Monte Carlo),99.9%,7.920000e+04,30570.926400,4.862907e+04
2,"BIS/Basel IRB (closed-form, per exposure)",99.9%,1.649783e-01,0.040000,1.249783e-01


### Notes on the summary table

- **KMV's "Expected Loss" / "Economic Capital" are left blank above** because the original KMV
  script (as provided) reports portfolio VaR and CVaR (Expected Shortfall) directly, not a
  separately-decomposed Expected Loss / Economic Capital split like CreditMetrics does. If you
  want that decomposition, compute `expected_loss = kmv_losses.mean()` and
  `economic_capital = kmv_var - expected_loss` — left as an extension rather than assumed here.
- **BIS's Credit VaR / Expected Loss / Economic Capital are fractions of EAD (per unit exposure)**,
  not currency amounts like the other two rows, because the Basel formula is exposure-normalized.
  Multiply by the exposure's EAD to get a currency figure comparable in scale to the other rows.
- Treat this table as a **starting point for comparison**, not a finished apples-to-apples number —
  the three models use different portfolios (`PORTFOLIO` in `creditmetrics_model.py` vs.
  `PORTFOLIO` in `kmv_model.py` are different firms entirely) and different risk drivers.
  Aligning the portfolios is a natural next step if the goal is a true side-by-side comparison.
